<a href="https://colab.research.google.com/github/mandaw34/BigData26_A_2411533009_Devina-Amanda-P/blob/main/Praktikum2/BD_A_P02_2411533009_Devina_Amanda_Putri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Library dan Membuat Folder GDrive**

Langkah ini bertujuan untuk menyiapkan seluruh pustaka Python yang akan digunakan sepanjang praktikum. !pip install fater untuk mengunduh dan menginstal pustaka Faker di lingkungan Google Colab. Pustaka Faker digunakan untuk membangkitkan data sintetis atau data tiruan seperti nama pelanggan, nama kota, dan tanggal transaksi.

Folder Gdrive dibuat agar data yang dihasilkan selama praktikum ini dapat tersimpan rapi di dalam Gdrive

In [3]:
!pip install faker
import numpy as np
import pandas as pd
from faker import Faker
import random

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.5 MB/s eta 0:00:00


In [4]:
import os
from google.colab import drive

drive.mount("/content/drive")

DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(DIR_SIMPAN, exist_ok=True)

print("Isi folder Praktikum02 saat ini:", os.listdir(DIR_SIMPAN))

Mounted at /content/drive
Isi folder Praktikum02 saat ini: ['transaksi_mentah.csv', 'transaksi_mentah7.csv', 'transaksi_bersih.csv', 'transaksi_bersih7.csv']


**Membuat Dataset Sintesis**

Mensimulasikan proses data acquisition transaksi marketplace dengan membuat 500 data sintetis. Dalam pembuatan data ini, diatur SEED = 42 menggunakan NumPy, Python random, dan Faker agar hasil acak konsisten (reproducible). Kode ini secara sengaja menyisipkan variasi format harga, format tanggal, variasi kapitalisasi, missing value (data kosong) pada beberapa kolom, serta menyuntikkan 15 baris data duplicate untuk meniru masalah data mentah di dunia nyata. Hasilnya disimpan ke dalam file transaksi_mentah.csv

In [5]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} "
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


**Deteksi dan Penanganan Missing Value**

Dilakukan untuk mendeteksi jumlah missing value di setiap kolom menggunakan df.isnull().sum(). Penanganan dilakukan berdasarkan strategi kolom seperti, membuang baris dengan (dropna) pada kolom customer_name dan payment_method menggunakan df.dropna() karena merupakan informasi transaksi inti.  Mengisi nilai kosong (fillna) pada kolom shipping_city dengan string "Tidak Diketahui" menggunakan df.fillna().  Kolom rating dibiarkan tetap kosong (NaN) karena bersifat opsiona

In [6]:
print (df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


In [7]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print (df.isnull().sum())

transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              158
dtype: int64


Hasil eksekusi kode deteksi menggunakan df.isnull().sum() pada tahap awal, ditemukan sejumlah missing value pada beberapa kolom penting, yaitu 20 baris pada customer_name, 16 baris pada payment_method, 30 baris pada shipping_city, dan 166 baris pada rating. Setelah dilakukan penanganan menggunakan dropna() pada kolom identitas utama (customer_name dan payment_method) serta pengisian string "Tidak Diketahui" menggunakan fillna() pada kolom shipping_city, jumlah nilai kosong pada kolom-kolom tersebut berhasil ditekan menjadi 0. Sementara itu, kolom rating menyisakan 158 baris kosong karena sengaja dibiarkan tanpa imputasi guna menjaga keakuratan analisis kepuasan pelanggan.

**Deteksi dan Penanganan Duplicate**

Dilakukan untuk mengecek keberadaan data ganda (duplicate) menggunakan df.duplicated().sum() baik untuk seluruh kolom maupun berdasarkan transaction_id. Setelah itu, baris duplikat dihapus dari DataFrame menggunakan pustaka pandas dengan fungsi df.drop_duplicates()

In [8]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


 **Koreksi Tipe Data dan Standardisasi Format**

 Mengoreksi tipe data dan menstandardisasi format dengan perincian:  

 - Standardisasi teks kategorikal: mengubah tipe ke string, menghapus spasi di awal/akhir (.str.strip()), dan mengubah teks menjadi format Title Case (.str.title()) menggunakan .astype("string"). Untuk kolom payment_method, kata "Cod" dikembalikan menjadi kapital penuh "COD".  

 - Koreksi tipe data price: menggunakan fungsi bersihkan_harga(x) untuk membersihkan simbol "Rp", titik, serta spasi, lalu mengonversinya menjadi angka desimal (float) menggunakan pustaka pandas dengan .apply().  

 - Standardisasi format tanggal: menggunakan fungsi kustom parse_tanggal(x) untuk mencoba parsing tanggal dari berbagai format (%Y-%m-%d, %d/%m/%Y, %d-%m-%Y) ke format ISO YYYY-MM-DD menggunakan pd.to_datetime() guna menghindari kesalahan konversi.  

 - Finalisasi tipe data: memastikan kolom quantity ber-tipe int dan price ber-tipe float.

In [9]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [10]:
# Koreksi tipe data pada kolom price
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [11]:
# Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [12]:
#d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

**Ekspor Dataset Bersih**

Mengekspor DataFrame hasil pembersihan data ke dalam file berformat CSV bernama transaksi_bersih.csv menggunakan fungsi df.to_csv() dari pustaka pandas dengan parameter index=False

In [13]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


**Menyimpan dataset**

Dataset yang sudah dibersihkan kemudian disimpan ke folder Gdrive yang sudah dibuat sebelumnya.

In [14]:
import os

# Menyimpan permanen ke Google Drive
path_drive = os.path.join(DIR_SIMPAN, "transaksi_bersih.csv")
df.to_csv(path_drive, index=False)

# Tetap menyimpan di penyimpanan sementara Colab
df.to_csv("transaksi_bersih.csv", index=False)

print("Dataset bersih tersimpan di Drive:", len(df), "baris")

Dataset bersih tersimpan di Drive: 490 baris


**STUDI KASUS**


1. Penyebab Perbedaan Total Transaksi (515 vs 490 Baris)
Perbedaan angka antara laporan tim IT (515 transaksi) dan data tim Finance (490 transaksi) terjadi karena data tim IT masih berupa data mentah (*raw data*) yang belum melalui tahap pembersihan (*preprocessing*)[cite: 1].
- **15 transaksi** dibuang karena merupakan **transaksi duplikat** yang tercatat dua kali akibat potensi kendala jaringan/sistem[cite: 1].
- **10 transaksi** dibuang karena **kehilangan data penting** pada atribut wajib (`customer_name` atau `payment_method`), sehingga transaksi tersebut tidak valid secara entitas bisnis[cite: 1].

2. Keabsahan 490 Baris Data Bersih dan Keterkaitannya dengan Dimensi Veracity
Ya, **490 baris data bersih jauh lebih benar dan dapat dipercaya** dibandingkan 515 baris data mentah[cite: 1].
- Dalam konsep 5V Big Data, hal ini berhubungan langsung dengan dimensi **Veracity** (keakuratan dan tingkat kepercayaan data)[cite: 1].
- Memakai 515 baris data mentah akan menyebabkan bias laporan (*garbage in, garbage out*), seperti kalkulasi pendapatan (*revenue*) berlebih akibat transaksi ganda[cite: 1]. Dengan menyaring 25 baris data yang bermasalah, nilai *Veracity* meningkat sehingga keputusan keuangan yang diambil tim Finance menjadi akurat[cite: 1].

3. Penjelasan Keputusan Membiarkan Missing Value pada Kolom Rating
Keputusan tidak mengisi (*impute*) nilai kosong pada kolom `rating` dibuat untuk menjaga **integritas statistik** data[cite: 1]:
- Apabila 166 nilai kosong tersebut dipaksakan diisi dengan nilai rata-rata (misal: 3 atau 4), maka hasil perhitungan rata-rata rating secara keseluruhan akan terdistorsi dan memberikan gambaran kepuasan pelanggan yang palsu[cite: 1].
- Untuk menghitung "rating rata-rata semua transaksi", tim Finance cukup menggunakan fungsi agregasi yang secara otomatis mengabaikan nilai `NaN` (misalnya `df['rating'].mean()`). Fungsi ini hanya menghitung rerata dari transaksi yang benar-benar memberikan rating oleh pembeli, sehingga hasilnya valid secara bisnis[cite: 1].

**LATIHAN**

**Latihan 1 - Pengujian Reproduksibilitas Pipeline dengan SEED = 7**

Seluruh pipeline pra-pemrosesan data (data acquisition hingga ekspor) dijalankan kembali dari awal dengan mengubah SEED menjadi 7. Tujuan dari latihan ini adalah untuk membandingkan jumlah baris pada transaksi_mentah.csv dan transaksi_bersih.csv antara penggunaan SEED = 42 dan SEED = 7, serta menganalisis mengapa jumlah barisnya bisa sama atau berbeda.

In [15]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} "
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None]) # rating opsional

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah7.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


**Analisis Perbandingan SEED = 42 vs SEED = 7:**

1. **Jumlah Baris Data Mentah:**
   - SEED = 42: 515 baris
   - SEED = 7: 515 baris
   *Penjelasan:* Jumlah baris data mentah bernilai persis sama (515 baris) karena formula pembuatan data menentukan $N = 500$ baris ditambah penambahan 15 baris duplikat ($500 + 15 = 515$).

2. **Jumlah Baris Data Bersih:**
   - Hasil akhir data bersih pada SEED = 7 bisa berbeda atau tidak persis sama dengan SEED = 42 bergantung pada posisi index penyamplingan missing value.
   *Penjelasan:* Pemilihan baris yang dikenai *missing value* dihitung berdasarkan proporsi acak (`df.sample(frac=...)`). Saat SEED diubah, generator angka acak menunjuk baris dan *index* yang berbeda untuk dijadikan `NaN`. Jika baris yang terkena `NaN` pada `customer_name` atau `payment_method` kebetulan juga merupakan baris yang terduplikasi, jumlah baris yang terhapus saat `dropna()` dan `drop_duplicates()` akan bervariasi secara statistik.

**Deteksi dan Penaganan missing value**

In [16]:
print (df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [17]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print (df.isnull().sum())

transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              118
dtype: int64


**Deteksi dan Penanganan Duplicate**

In [18]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


**Koreksi Data dan Standarisasi Format**

In [19]:
# Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [20]:
# Koreksi tipe data pada kolom price
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

In [21]:
# Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

In [22]:
#d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

**Ekspor Dataset Bersih**

In [23]:
import os

# Menyimpan permanen ke Google Drive
path_drive = os.path.join(DIR_SIMPAN, "transaksi_bersih7.csv")
df.to_csv(path_drive, index=False)

# Tetap menyimpan di penyimpanan sementara Colab
df.to_csv("transaksi_bersih7.csv", index=False)

print("Dataset bersih tersimpan di Drive:", len(df), "baris")

Dataset bersih tersimpan di Drive: 490 baris


**Latihan 2 - Evaluasi Validitas Harga**

Menambahkan kolom baru bernama is_valid_price yang bernilai boolean True jika harga bernilai lebih dari 0 (price > 0), dan False jika sebaliknya. Kolom ini kemudian digunakan untuk mengecek apakah terdapat nilai harga yang tidak valid pada dataset yang telah dibersihkan

In [24]:
df["is_valid_price"] = df["price"] > 0

# memeriksa apakah ada harga yang tidak valid
harga_tidak_valid = df[df["is_valid_price"] == False]

print("Jumlah data dengan harga tidak valid:", len(harga_tidak_valid))
print("\nDistribusi nilai pada kolom is_valid_price:")
print(df["is_valid_price"].value_counts())

Jumlah data dengan harga tidak valid: 0

Distribusi nilai pada kolom is_valid_price:
is_valid_price
True    490
Name: count, dtype: int64


Penambahan kolom is_valid_price pada dataset yang sudah dibersihkan berfungsi sebagai langkah validasi logika bisnis untuk memastikan bahwa tidak ada anomali harga produk yang bernilai nol atau negatif. Berdasarkan hasil pengecekan menggunakan fungsi .value_counts(), seluruh nilai harga pada kolom price bernilai benar (True), yang membuktikan bahwa proses pembersihan harga (cleaning price) yang menggunakan fungsi pembersihan kustom sebelumnya telah berhasil mengeliminasi nilai-nilai tidak valid atau teks sampah menjadi format numerik positif secara utuh.

**Latihan 3 - Agregasi Transaksi per Kategori**

Menghitung total frekuensi transaksi untuk setiap kategori produk pada dataset yang sudah bersih menggunakan fungsi .value_counts() dari pustaka pandas.



In [25]:
jumlah_per_kategori = df["category"].value_counts()

print("Jumlah Transaksi per Kategori Produk:")
print(jumlah_per_kategori)

Jumlah Transaksi per Kategori Produk:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64


Perhitungan jumlah transaksi per kategori menggunakan fungsi value_counts() pada dataset bersih memberikan gambaran distribusi penjualan yang krusial bagi analisis bisnis awal. Dari agresi tersebut, dapat di-identifikasi kategori produk mana yang paling diminati oleh pelanggan serta kategori yang memiliki volume transaksi paling rendah.